In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import (f1_score, matthews_corrcoef,
                             average_precision_score, roc_auc_score,
                             classification_report)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('../data/train_final.csv')
test  = pd.read_csv('../data/test_final.csv')

X_train = train.drop(columns=['isFraud'])
y_train = train['isFraud']
X_test  = test.drop(columns=['isFraud'])
y_test  = test['isFraud']

print(f"Train: {X_train.shape}")
print(f"Test:  {X_test.shape}")

Train: (15624, 192)
Test:  (2000, 192)


In [3]:
def evaluate_model(name, y_true, y_pred, y_prob):
    f1    = f1_score(y_true, y_pred)
    mcc   = matthews_corrcoef(y_true, y_pred)
    auprc = average_precision_score(y_true, y_prob)
    auc   = roc_auc_score(y_true, y_prob)
    
    print(f"\n{'='*40}")
    print(f"Model: {name}")
    print(f"{'='*40}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"MCC:       {mcc:.4f}")
    print(f"AUPRC:     {auprc:.4f}")
    print(f"ROC-AUC:   {auc:.4f}")
    print(f"\n{classification_report(y_true, y_pred, target_names=['Legit','Fraud'])}")
    
    return {'model': name, 'F1': f1, 'MCC': mcc, 'AUPRC': auprc, 'AUC': auc}

In [4]:
print("Training XGBoost...")

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=1,
    random_state=42,
    eval_metric='aucpr',
    verbosity=0
)

xgb_model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              verbose=False)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

results_xgb = evaluate_model("XGBoost", y_test, y_pred_xgb, y_prob_xgb)

Training XGBoost...

Model: XGBoost
F1 Score:  0.3579
MCC:       0.4487
AUPRC:     0.5649
ROC-AUC:   0.8812

              precision    recall  f1-score   support

       Legit       0.97      1.00      0.98      1923
       Fraud       0.94      0.22      0.36        77

    accuracy                           0.97      2000
   macro avg       0.96      0.61      0.67      2000
weighted avg       0.97      0.97      0.96      2000



In [5]:
print("Training LightGBM...")

lgb_model = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

lgb_model.fit(X_train, y_train)

y_pred_lgb = lgb_model.predict(X_test)
y_prob_lgb = lgb_model.predict_proba(X_test)[:, 1]

results_lgb = evaluate_model("LightGBM", y_test, y_pred_lgb, y_prob_lgb)

Training LightGBM...

Model: LightGBM
F1 Score:  0.3958
MCC:       0.4894
AUPRC:     0.5574
ROC-AUC:   0.8543

              precision    recall  f1-score   support

       Legit       0.97      1.00      0.99      1923
       Fraud       1.00      0.25      0.40        77

    accuracy                           0.97      2000
   macro avg       0.99      0.62      0.69      2000
weighted avg       0.97      0.97      0.96      2000



In [ ]:
    # Load kết quả baseline cũ
baseline = pd.read_csv('../reports/baseline_results.csv')

# Thêm kết quả mới
new_results = pd.DataFrame([results_xgb, results_lgb])
all_results = pd.concat([baseline, new_results], ignore_index=True)

print("\n📊 Bảng so sánh TẤT CẢ model:")
print(all_results.to_string(index=False))

# Vẽ biểu đồ
metrics = ['F1', 'MCC', 'AUPRC', 'AUC']
x = np.arange(len(metrics))
width = 0.2
colors = ['steelblue', 'coral', 'green', 'purple']

fig, ax = plt.subplots(figsize=(12, 5))
for i, (_, row) in enumerate(all_results.iterrows()):
    ax.bar(x + i*width, row[metrics], width, 
           label=row['model'], color=colors[i])

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(metrics)
ax.set_ylabel('Score')
ax.set_title('So sánh tất cả models')
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../reports/all_models_comparison.png', dpi=80, bbox_inches='tight')
plt.close()

all_results.to_csv('../reports/all_results.csv', index=False)
print("\n✅ Đã lưu kết quả và biểu đồ!")